# Práctica 1: Exploración de Niveles del Lenguaje

**Nombre:** Omar Fernando Gramer Muñoz<br>
**Materia:** Lingüística Computacional<br>
**Matrícula:** 419003698

In [5]:
# Importaciones
import http
from collections import defaultdict

import pandas as pd
import requests as r

from rich import print as rprint
from rich.columns import Columns
from rich.panel import Panel
from rich.text import Text

## **Fonética**
___

#### **Con base en el sistema de búsqueda visto en la práctica 1 dónde se recibe una palabra ortográfica y devuelve sus transcripciones fonológicas, proponga una solución para los casos en que la palabra buscada no se encuentra en el lexicón/diccionario**

### **¿Cómo devolver o aproximar su transcripción fonológica?**

El sistema de búsqueda de la práctica devuelve una lista vacía cuando 
la palabra no existe en el lexicón. 

Una alternativa es aprovechar que el español tiene una correspondencia 
**grafema-fonema muy regular**: en la mayoría de los casos, sabemos 
exactamente cómo pronunciar una letra según su contexto. A diferencia 
del inglés donde "gh" puede sonar como /f/ (enough), /ɡ/ (ghost) o 
ser muda (night)— el español sigue reglas mucho más predecibles.

Esto nos permite construir un sistema **G2P (Grapheme-to-Phoneme)**:
un conjunto ordenado de reglas que convierte secuencias de letras en 
símbolos IPA. Las reglas más específicas (dígrafos como "ch", "ll", 
"rr") se aplican antes que las individuales para evitar ambigüedades.

**La estrategía será:**

1. Buscar la palabra en el lexicón (comportamiento original)
2. Si se encuentra entonces devolver la transcripción validada
3. Si no se encuentra aplicar las reglas G2P para generar 
   una transcripción aproximada

De esta forma el sistema nunca devuelve una lista vacía.

In [6]:
IPA_URL = "https://raw.githubusercontent.com/open-dict-data/ipa-dict/master/data/{lang}.txt"

In [13]:
# Funciones de descarga y parseo del dataset ( tomadas del notebook de la clase )
def download_ipa_corpus(iso_lang: str) -> str:
    print(f"Downloading {iso_lang}", end="::")
    response = r.get(IPA_URL.format(lang=iso_lang))
    status_code = response.status_code
    print(f"status={status_code}")
    if status_code != http.HTTPStatus.OK:
        print(f"ERROR on {iso_lang} :(")
        return ""
    return response.text

def parse_response(response: str) -> dict:
    ipa_list = response.rstrip().split("\n")
    result = {}
    for item in ipa_list:
        if item == '':
            continue
        item_list = item.split("\t")
        result[item_list[0]] = item_list[1]
    return result

In [14]:
# Descargar el dataset de español mexicano
dataset_es_mx = parse_response(download_ipa_corpus("es_MX"))

In [19]:
# Obviamente utlicé Claude para esto a nivel de 'Delegación supervisada', 
# no hay forma de que escribiera estas reglas a mano sin que se me escapara alguna o estuviera incorrecta.
# Desde luego que reviso el código, lo entiendo y hago las modificaciones que sienta pertinentes.

# Reglas grafema → fonema para español mexicano
# Se aplican en orden (las más específicas primero)
def g2p_es(word: str) -> str:
    text = word.lower()
    result = ""
    i = 0
    while i < len(text):
        char = text[i]
        next_char = text[i+1] if i+1 < len(text) else ""
        
        # Dígrafos (verificar primero)
        digraph = char + next_char
        if digraph == "ch":
            result += "tʃ"; i += 2
        elif digraph == "ll":
            result += "ʝ"; i += 2
        elif digraph == "rr":
            result += "r"; i += 2
        elif digraph == "qu":
            result += "k"; i += 2
        elif digraph in ("ge", "gi"):
            result += "x"; i += 1  # solo consumimos g, la vocal sigue
        elif digraph in ("gue", "gui"):
            result += "ɡ"; i += 2
        
        # Reglas contextuales para 'c'
        elif char == "c" and next_char in ("e", "i"):
            result += "s"; i += 1  # ce, ci → s
        elif char == "c":
            result += "k"; i += 1  # ca, co, cu → k
        
        # Reglas contextuales para 'g'
        elif char == "g" and next_char in ("e", "i"):
            result += "x"; i += 1  # ge, gi → x
        elif char == "g":
            result += "ɡ"; i += 1  # ga, go, gu → ɡ
        
        # Reglas simples
        elif char == "h":
            i += 1  # muda
        elif char == "j":
            result += "x"; i += 1
        elif char == "r" and i == 0:
            result += "r"; i += 1  # r inicial es vibrante múltiple
        elif char == "r":
            result += "ɾ"; i += 1  # r intervocálica es vibrante simple
        elif char == "ñ":
            result += "ɲ"; i += 1
        elif char in ("b", "v"):
            result += "β"; i += 1
        elif char == "y":
            result += "ʝ"; i += 1
        elif char == "z":
            result += "s"; i += 1
        elif char == "x":
            result += "ks"; i += 1
        else:
            # Vocales y consonantes directas
            direct = {"a":"a","e":"e","i":"i","o":"o","u":"u",
                      "á":"a","é":"e","í":"i","ó":"o","ú":"u",
                      "p":"p","t":"t","k":"k","f":"f","s":"s",
                      "l":"l","m":"m","n":"n","d":"ð","w":"w"}
            result += direct.get(char, char)
            i += 1
    
    return f"/{result}/"

### **Reutiliza el sistema de búsqueda visto en clase y mejóralo con esta funcionalidad**

In [20]:
def get_ipa_transcriptions_improved(word: str, dataset: dict) -> tuple[list[str], str]:
    """
    Versión mejorada del sistema de búsqueda.
    
    Primero busca en el lexicón. Si no encuentra la palabra,
    genera una transcripción aproximada con reglas G2P.
    
    Parameters
    ----------
    word : str
        Palabra a buscar
    dataset : dict
        Diccionario IPA del idioma
    
    Returns
    -------
    tuple[list[str], str]
        (transcripciones, fuente) donde fuente es 
        'lexicon' o 'g2p'
    """
    result = dataset.get(word.lower(), "")
    if result:
        return result.split(", "), "lexicon"
    else:
        return [g2p_es(word)], "g2p"

### **Muestra al menos tres ejemplos**

In [24]:
# ============================================================
# Ejemplos de uso del sistema mejorado
# ============================================================

test_words = [
    # Palabra conocida (en lexicón)
    "mayonesa",
    # Neologismo (OOV)
    "tuitear",
    # Nombre propio (OOV)
    "xochimilco",
    # Palabra con ortografía especial (OOV)
    "whatsappear",
    # Conocida
    "corazón",
    # Nombre propio (OOV)
    "tlalpan",
    # Neologismo (OOV)
    "googlear",
    # Neologismo (OOV)
    "chatgptear",
    # Palabra conocida
    "muchacho",
    # Me la inventé
    "muchachismo"
]

print("=" * 55)
print(f"{'Palabra':<15} {'Transcripción':<20} {'Fuente'}")
print("=" * 55)

for word in test_words:
    transcriptions, source = get_ipa_transcriptions_improved(word, dataset_es_mx)
    icon = "📖" if source == "lexicon" else "🔧"
    print(f"{word:<15} {', '.join(transcriptions):<20} {icon} {source}")

print("=" * 55)

Palabra         Transcripción        Fuente
mayonesa        /maʝonesa/           📖 lexicon
tuitear         /tuiteaɾ/            🔧 g2p
xochimilco      /ksotʃimilko/        🔧 g2p
whatsappear     /watsappeaɾ/         🔧 g2p
corazón         /koɾaˈson/           📖 lexicon
tlalpan         /tlalpan/            🔧 g2p
googlear        /ɡooɡleaɾ/           🔧 g2p
chatgptear      /tʃatɡpteaɾ/         🔧 g2p
muchacho        /mutʃatʃo/           📖 lexicon
muchachismo     /mutʃatʃismo/        🔧 g2p


## **Morfología**
***

### **Elige tres lenguas del corpus que pertenezcan a familias lingüísticas distintas**

In [26]:
import requests

def check_unimorph(lang: str) -> tuple[bool, int]:
    """Verifica si una lengua existe en UniMorph y cuántas entradas tiene."""
    url = f"https://raw.githubusercontent.com/unimorph/{lang}/main/{lang}"
    response = requests.get(url)
    if response.status_code == 200:
        entries = [l for l in response.text.strip().split("\n") 
                   if l and not l.startswith("#")]
        return True, len(entries)
    return False, 0

candidatos = {
    "Japonés":   "jpn",
    "Swahili":   "swa",
    "Georgiano": "kat",
}

print(f"{'Lengua':<12} {'Código':<8} {'Disponible':<12} {'Entradas'}")
print("=" * 45)
for nombre, codigo in candidatos.items():
    disponible, entradas = check_unimorph(codigo)
    icono = "✅" if disponible else "❌"
    print(f"{nombre:<12} {codigo:<8} {icono:<12} {entradas:,}" if disponible 
          else f"{nombre:<12} {codigo:<8} {icono}")

Lengua       Código   Disponible   Entradas
Japonés      jpn      ✅            12,687
Swahili      swa      ❌
Georgiano    kat      ❌
